In [15]:
import dspy
from typing import List, Optional, Literal
from pydantic import BaseModel,Field


In [16]:
from openai import OpenAI
import os
import dspy

lm = dspy.LM(
    model="openai/gpt-5-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)

In [17]:

# Structured output for a single policy
class ExtractedPolicy(BaseModel):
    # Core extraction
    policy_statement: str = Field("A concise, self-contained summary of the policy. Rewrite the original text into a clear 'commitment' statement if necessary.")  # Cleaned, self-contained policy statement
    verbatim_text: str  = Field("Details must be quoted verbatim from the source (no paraphrasing), try to include 2-3 sentences") 

    # Soundness indicators, include these here so that we don't lose info on text
    has_quantifiable_target: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the target (e.g., '40% reduction by 2030').")  # Whether it includes quantifiable targets
    has_timeline: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the timeline (e.g., 'by 2030').")
    has_binding_mechanism: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the binding mechanism (e.g., 'legal mandate').")
    has_spatial_specificity: str = Field("Indicate 'Yes' or 'No'. If 'Yes', provide details on the spatial specificity (e.g., 'national', 'state-level').")
    
    # Reasoning trace, tbh purely research based
    extraction_rationale: str = Field("Why this qualifies as a policy, based on the criteria provided in the initial prompt, any flags I should watch out for")# might need to redefine this




In [18]:
class DocumentMetadata(BaseModel):
    country: str 
    state_or_province: Optional[str]
    city: Optional[str] 


In [19]:
class PolicyExtractionSignature(dspy.Signature):
    """
    Extract climate policies from document text.
    
    DEFINITION OF A POLICY
    A policy is a STATED COMMITMENT by a governing body to achieve a defined 
    outcome through deliberate action, resource allocation, or regulatory change.
    
    A policy is NOT:
    - Background information or problem descriptions
    - Statements of current conditions
    - Aspirations without any specified action
    - Descriptions of what other actors might do
    
    WHAT MAKES SOMETHING EXTRACTABLE
    Extract a statement as a policy if it contains AT LEAST ONE of:
    
    1. QUANTIFIABLE TARGET: Numbers with units and/or deadlines
        -"Reduce emissions 40% by 2030"
       -"Install 5GW of solar capacity"
       -"Retrofit 10,000 buildings"
    
    2. BINDING MECHANISM: Legal or regulatory force
       -"Mandatory building codes requiring..."
       -"Ban on single-use plastics"
       -"Carbon tax of $25/ton"
    
    3. SPECIFIC INTERVENTION: Named program, technology, or action
       -"National E-Mobility Program"
       -"Phase-out of coal-fired power plants"
       -"Expansion of bus rapid transit network"
    
    4. RESOURCE ALLOCATION: Committed funding or investment
       -"$500M allocated for renewable energy"
       -"Dedicated climate adaptation fund"
    
    EDGE CASES: EXTRACT BUT FLAG
    - Vague commitments ("promote", "encourage", "explore") — extract if 
      they reference a specific sector or mechanism, note vagueness
    - Strategies/roadmaps without targets — extract if they name concrete 
      action areas
    - Conditional commitments ("subject to international funding") — extract,
      note conditionality
    
    DO NOT EXTRACT
    - Pure context: "Climate change threatens coastal areas"
    - Current state: "The country currently has 2GW of wind capacity"  
    - Process statements: "Stakeholder consultations will be held"
    - Vague aspirations with no anchor: "We aspire to a green future"
    
 
    If no policies are found in the chunk, return an empty list.
    """
    document_text: str = dspy.InputField(
        desc="Text extracted from a climate policy document (NDC, action plan, etc.)"
    )
    document_metadata: DocumentMetadata = dspy.InputField(
        desc="Document name, country, year, and any known section context"
    )
    
    policies: List[ExtractedPolicy] = dspy.OutputField(
        desc="List of extracted policies with metadata; empty list if none found"
    )


In [20]:
class PolicyExtractor(dspy.Module):
    """
    Extracts structured policy objects from document text.
    
    Designed to run on chunks/sections of a larger document.
    Handles the full extraction task: identification, extraction, 
    and preliminary soundness assessment.
    """
    
    def __init__(self):
        self.extract = dspy.ChainOfThought(PolicyExtractionSignature)
    
    def forward(self, document_text: str, document_metadata: str = "") -> List[ExtractedPolicy]:
        result = self.extract(
            document_text=document_text,
            document_metadata=document_metadata
        )
        return result.policies

In [8]:
from docling.document_converter import DocumentConverter
converter= DocumentConverter()

In [10]:
# documents = converter.convert(src)
key_to_pdf = {
"Seattle":"pdfs/Seattle.pdf",

"Las Vegas":"pdfs/CLV.pdf",
"Miami-Dade":"pdfs/MIAMI_DADE.pdf",
"Hiroshima":"pdfs/HIROSHIMA.pdf",
"Geneva":"pdfs/city_of_geneva.pdf",
"Porutgal":"pdfs/2021 Portugal ADCOM_UNFCCC.pdf"
}

keys_to_details ={
    "Seattle":['United States','Washington'],
    "Las Vegas":['United States','Nevada'],
    "Miami-Dade":["United States","Florida"],
    "Hiroshima":["Japan","N/A"],
    "Geneva":["Switzerland","N/A"],
    "Portugal":["Portugal","N/A"]
}

In [11]:
d_metadata_dict = {
    key: DocumentMetadata(
        country=details[0],
        state_or_province=details[1] if details[1] != "N/A" else None,
        city=key if key not in ["Portugal"] else None
    )
    for key, details in keys_to_details.items()
}


In [12]:
src = key_to_pdf["Seattle"]
d_metadata = d_metadata_dict["Seattle"]

In [13]:
document = converter.convert(src)

2026-01-20 13:27:00,591 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-20 13:27:00,694 - INFO - Going to convert document batch...
2026-01-20 13:27:00,695 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-01-20 13:27:00,717 - INFO - Loading plugin 'docling_defaults'
2026-01-20 13:27:00,720 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-01-20 13:27:00,725 - INFO - Loading plugin 'docling_defaults'
2026-01-20 13:27:00,730 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-01-20 13:27:02,394 - INFO - Auto OCR model selected ocrmac.
2026-01-20 13:27:02,399 - INFO - Loading plugin 'docling_defaults'
2026-01-20 13:27:02,401 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2026-01-20 13:27:04,164 - INFO - Accelerator device: 'mps'
2026-01-20 13:27:06,361 - INFO - Loading plugin 'docling_defaul

In [15]:
markdown_version = document.document.export_to_markdown()


In [16]:
policy_extractor = PolicyExtractor()
extracted_policies = policy_extractor(
    document_text=markdown_version,
    document_metadata=d_metadata
)


In [27]:
with open("extracted_policies_seattle.json", "w") as f:
    import json
    json.dump([policy.dict() for policy in extracted_policies], f, indent=2)

/var/folders/f5/bb79s3gs38q_wp4_2kvwbz9w0000gn/T/ipykernel_93596/1416483033.py:3: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  json.dump([policy.dict() for policy in extracted_policies], f, indent=2)


In [21]:
class PolicyValidationSignature(dspy.Signature):
    """Evaluate whether a climate policy statement is VALID (actionable and measurable) 
    or NON-SOUND (vague and performative) based on the Policy Extraction & Validation Protocol.
    
    A VALID policy must demonstrate:
    1. Quantifiable Targets: Includes numbers, dates, and units
    2. Binding Mechanisms: Backed by law, regulation, or formal mandate
    3. Spatial Specificity: Identifies where impact will occur
    4. Technological Shift: Specifies transition from one state to another
    
    A NON-SOUND policy exhibits:
    1. Process without Outcome: Focuses on meetings/dialogue rather than results
    2. Aspirational Language: Uses "aim to," "explore," "strive" without roadmap
    3. Lack of Baseline: Offers change without stating starting point
    4. Administrative Maintenance: Standard operations rebranded as climate action



    it doesnt have to have all valid markers, but it should be valid in that it exhibits valid traits strongly and non-sound ones weakly or not at all
    """
    
    policy_statement: str = dspy.InputField(desc="The climate policy statement to evaluate")
    verbatim_text: str = dspy.InputField(desc="The original verbatim text from the source document")
    has_quantifiable_target: str = dspy.InputField(desc="Whether policy includes quantifiable targets")
    has_timeline: str = dspy.InputField(desc="Whether policy includes specific timeline/deadline")
    has_binding_mechanism: str = dspy.InputField(desc="Whether policy has legal/regulatory backing")
    has_spatial_specificity: str = dspy.InputField(desc="Whether policy identifies specific location/scope")
    
    validation_result: Literal["VALID", "NON-SOUND"] = dspy.OutputField(
        desc="Final classification: VALID (actionable/measurable) or NON-SOUND (vague/performative)"
    )
    
    confidence_score: float = dspy.OutputField(
        desc="Confidence in validation (0.0 to 1.0), where 1.0 is highest confidence"
    )
    
    reasoning: str = dspy.OutputField(
        desc="Detailed explanation referencing specific validation criteria and red flags"
    )
    
    weak_signals: str = dspy.OutputField(
        desc="Any red flag keywords detected (e.g., 'promote', 'explore', 'awareness', 'workshop')"
    )
    
    strong_signals: str = dspy.OutputField(
        desc="Any valid markers detected (e.g., percentages, MW, hectares, 'mandate', 'install', 'phase-out')"
    )
    final_verdict: bool = dspy.OutputField(
        desc="Based on the confidence score and validation result, is this policy likely to be sound?"
    )

In [22]:
class PolicyValidator(dspy.Module):
    """DSPy module for validating climate policy soundness"""
    
    def __init__(self):
        super().__init__()
        self.validate = dspy.ChainOfThought(PolicyValidationSignature)
    
    def forward(self, policy_data: dict):
        """
        Validate a policy using the extraction framework
        
        Args:
            policy_data: Dictionary containing policy fields
            
        Returns:
            DSPy prediction with validation results
        """
        result = self.validate(
            policy_statement=policy_data.get("policy_statement", ""),
            verbatim_text=policy_data.get("verbatim_text", ""),
            has_quantifiable_target=policy_data.get("has_quantifiable_target", "Unknown"),
            has_timeline=policy_data.get("has_timeline", "Unknown"),
            has_binding_mechanism=policy_data.get("has_binding_mechanism", "Unknown"),
            has_spatial_specificity=policy_data.get("has_spatial_specificity", "Unknown"),
            final_verdict=policy_data.get("final_verdict", False)
        )
        
        return result


In [38]:
validator = PolicyValidator()
results = []
for policy in extracted_policies:
    policy_dict = policy.dict()
    validation_result = validator.forward(policy_dict)
    results.append(validation_result)


/var/folders/f5/bb79s3gs38q_wp4_2kvwbz9w0000gn/T/ipykernel_26715/4266502906.py:4: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  policy_dict = policy.dict()
2026/01/20 13:42:48 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026-01-20 13:42:52,622 - INFO - Retrying request to /chat/completions in 0.492194 seconds
2026/01/20 13:43:17 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 13:43:36 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 13:43:49 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is di

In [39]:
with open("validation_results_seattle.json", "w") as f:
    import json
    # Manually create dicts from the prediction objects to avoid serialization issues
    json.dump([{
        'validation_result': result.validation_result,
        'confidence_score': result.confidence_score,
        'reasoning': result.reasoning,
        'weak_signals': result.weak_signals,
        'strong_signals': result.strong_signals,
        'final_verdict': result.final_verdict
    } for result in results], f, indent=2)

In [40]:
# Run validation and combine results
validator = PolicyValidator()
valid_policies = []
reasoning_traces = []

for idx, policy in enumerate(extracted_policies):
    policy_dict = policy.dict()
    validation_result = validator.forward(policy_dict)
    
    # Only keep policies marked as VALID with final_verdict=True
    if validation_result.validation_result == "VALID" and validation_result.final_verdict:
        # Combine original policy fields with validation verdict
        combined_policy = {
            **policy_dict,  # All original fields from ExtractedPolicy
            'validation_status': validation_result.validation_result,
            'confidence_score': validation_result.confidence_score,
            'final_verdict': validation_result.final_verdict
        }
        valid_policies.append(combined_policy)
    
    # Store reasoning trace separately for all policies (valid and non-sound)
    reasoning_traces.append({
        'policy_index': idx,
        'policy_statement': policy_dict['policy_statement'],
        'validation_result': validation_result.validation_result,
        'confidence_score': validation_result.confidence_score,
        'reasoning': validation_result.reasoning,
        'weak_signals': validation_result.weak_signals,
        'strong_signals': validation_result.strong_signals,
        'final_verdict': validation_result.final_verdict
    })

# Save valid policies with original fields
with open("valid_policies_seattle.json", "w") as f:
    import json
    json.dump(valid_policies, f, indent=2)

# Save reasoning traces separately
with open("reasoning_traces_seattle.json", "w") as f:
    json.dump(reasoning_traces, f, indent=2)

print(f"Total extracted policies: {len(extracted_policies)}")
print(f"Valid policies: {len(valid_policies)}")
print(f"Filtered out: {len(extracted_policies) - len(valid_policies)}")

/var/folders/f5/bb79s3gs38q_wp4_2kvwbz9w0000gn/T/ipykernel_26715/1932984991.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  policy_dict = policy.dict()
2026/01/20 13:55:51 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 13:55:51 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 13:55:51 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 13:55:51 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/20 13:55:51 WARNING dspy.primitives.modu

Total extracted policies: 18
Valid policies: 10
Filtered out: 8


In [24]:
# Import the classification classes and functions at the top of your notebook
from typing import List, Optional
import json

class PolicyClassificationSignature(dspy.Signature):
    """Classify a climate policy into one or more categories based on its primary causal pathway.
    
    CATEGORY DEFINITIONS:
    
    1. MITIGATION - Acts on the climate system itself
       
       Definition: Mitigation policies are designed to influence the climate system by changing the amount 
       of greenhouse gases humans release into the atmosphere or by increasing the ability of natural or 
       engineered systems to remove those gases from the atmosphere. These are forward-looking policies 
       concerned with preventing future climate change rather than responding to impacts already occurring.
       
       Primary pathway: Human activity leads to DECREASED GHG emissions OR INCREASED carbon sinks, 
       which leads to DECREASED atmospheric forcing
       
       What to look for:
       - Direct or indirect emissions reduction targets
       - Energy system transformation (fossil fuels to renewables)
       - Decarbonization targets or net-zero commitments
       - Carbon sequestration framed in climate terms
       - Success measured by emissions reduced or carbon sequestered
       
       Typical mechanisms: renewable energy mandates, electrification (transport/buildings), 
       carbon pricing/offsets/caps, fuel switching, carbon capture and storage (CCS)
       
    2. ADAPTATION - Responds to climate impacts and builds resilience
       
       Definition: Adaptation policies reduce the negative consequences of climate change by helping 
       people, ecosystems, and infrastructure cope with current or expected climate impacts. Unlike 
       mitigation, adaptation does not attempt to slow climate change itself—it assumes climate change 
       is happening and focuses on reducing harm. Resilience is core: the capacity to continue functioning 
       under climate stress and recover quickly after disruption.
       
       Primary pathway: Climate hazard leads to DECREASED exposure OR DECREASED sensitivity OR 
       INCREASED adaptive capacity
       
       What to look for:
       - Explicit reference to climate impacts (heat, flooding, drought, sea-level rise, storms)
       - Resilience, preparedness, or risk reduction language
       - Protection of people, assets, or systems from future climate conditions
       - Success measured by reduced losses, maintained services, avoided disruption, increased preparedness
       
       Typical mechanisms: flood defenses, heat action plans, climate-resilient infrastructure, 
       emergency response capacity, insurance/risk-transfer instruments, early warning systems
       
    3. RESOURCE EFFICIENCY - Optimizes resource use regardless of climate
       
       Definition: Resource efficiency policies deliver the same level of service, output, or quality 
       of life while using fewer physical resources (energy, water, materials, land). The core idea is 
       that systems are inefficient and consume more than necessary. These policies can be fully understood 
       and justified even in the absence of climate change concerns—efficiency is the goal, not climate impact.
       
       Primary pathway: Same service leads to DECREASED energy/water/material input OR DECREASED waste output
       
       What to look for:
       - Efficiency improvements that don't require climate justification
       - Optimization of inputs, flows, or lifecycles
       - Performance-based standards and benchmarks
       - Success measured by resource savings per unit of output
       - Emissions reduction is a secondary consequence, NOT the primary goal
       
       Typical mechanisms: energy efficiency codes, water efficiency standards, circular economy policies, 
       waste reduction/recycling mandates, industrial process optimization, building performance standards
       
    4. NATURE-BASED SOLUTIONS (NbS) - Uses ecosystems as infrastructure
       
       Definition: Nature-based solutions intentionally use natural systems or ecological processes to 
       address environmental, climate, or societal challenges. Effectiveness depends on the functioning 
       of living systems rather than solely on engineered infrastructure. These are defined by their 
       MECHANISM (ecosystems), not their outcome. If removing the ecological element would fundamentally 
       break the policy's effectiveness, it's NbS.
       
       Primary pathway: Ecosystem protection/restoration leads to climate mitigation OR resilience 
       OR broader co-benefits
       
       What to look for:
       - Explicit use of ecosystems as infrastructure
       - Restoration, conservation, or enhancement of natural systems
       - Solutions that rely on biological processes (carbon uptake, water absorption, cooling)
       - Policy effectiveness depends on living systems functioning properly
       
       Typical mechanisms: urban forests, wetland restoration, green roofs and corridors, 
       mangroves, coastal dunes, riparian buffers, soil restoration
    
    CLASSIFICATION RULES:
    
    Rule 1: Primary mechanism > secondary benefits
    - Classify by what the policy DOES, not by claimed co-benefits
    - Example: "Energy efficiency to reduce emissions" → Resource Efficiency (NOT Mitigation)
    
    Rule 2: Climate system vs climate impacts
    - Acts on emissions/carbon cycle → Mitigation
    - Acts on hazards/vulnerability → Adaptation
    
    Rule 3: Natural vs engineered pathways
    - Ecosystem-mediated → NbS
    - Technological/infrastructural → other categories
    
    Rule 4: Multi-label is allowed
    - Policies can be Mitigation + NbS, Adaptation + NbS, etc.
    - But the PRIMARY label must be identifiable
    
    EDGE CASES:
    
    Green roofs:
    - Urban heat exposure/stormwater management → Adaptation
    - Ecosystem restoration → NbS
    - Carbon storage → Mitigation (secondary)
    
    Passive cooling/shading:
    - Protecting occupants during heatwaves → Adaptation
    - Reducing energy demand → Resource Efficiency
    
    District energy:
    - Decarbonizing heating/cooling → Mitigation
    - Energy reliability during extreme weather → Adaptation
    
    Floodable parks/wetlands:
    - Reducing flood risk through natural absorption → NbS + Adaptation
    - Biodiversity enhancement (no climate risk) → NbS only
    
    Infrastructure hardening:
    - Strengthening grids/systems for storms/heat → Adaptation
    - Not mitigation unless emissions reduction is main objective
    """
    
    policy_statement: str = dspy.InputField(desc="The climate policy statement to classify")
    verbatim_text: str = dspy.InputField(desc="Original verbatim text from source document")
    context: str = dspy.InputField(desc="Additional context about the policy (optional)", default="")
    
    primary_category: str = dspy.OutputField(
        desc="The PRIMARY category: Mitigation, Adaptation, Resource Efficiency, or Nature-Based Solutions"
    )
    
    secondary_categories: str = dspy.OutputField(
        desc="Additional applicable categories as comma-separated list, or 'None' if single-category"
    )
    
    primary_causal_pathway: str = dspy.OutputField(
        desc="The policy's primary causal pathway (e.g., 'Climate hazard → ↓ exposure')"
    )
    
    key_indicators: str = dspy.OutputField(
        desc="Specific words/phrases that signaled this classification (e.g., 'flood defenses', 'renewables mandate', 'ecosystem restoration')"
    )
    
    policy_mechanisms: str = dspy.OutputField(
        desc="The specific mechanisms used (e.g., 'renewable energy mandates', 'heat action plans', 'wetland restoration')"
    )
    
    classification_reasoning: str = dspy.OutputField(
        desc="Detailed step-by-step explanation of why this classification was chosen, referencing specific rules"
    )
    
    confidence_score: float = dspy.OutputField(
        desc="Confidence in classification (0.0 to 1.0)"
    )
    
    edge_case_notes: str = dspy.OutputField(
        desc="Any edge case considerations or ambiguities encountered, or 'None'"
    )
    
    co_benefits: str = dspy.OutputField(
        desc="Secondary benefits mentioned but not used for primary classification (e.g., 'emissions reduction', 'biodiversity'), or 'None'"
    )


class PolicyClassifier(dspy.Module):
    """DSPy module for climate policy classification"""
    
    def __init__(self):
        super().__init__()
        self.classify = dspy.ChainOfThought(PolicyClassificationSignature)
    
    def forward(self, policy_data: dict):
        result = self.classify(
            policy_statement=policy_data.get("policy_statement", ""),
            verbatim_text=policy_data.get("verbatim_text", "")
        )
        return result


# After your soundness validation, add this cell:
# Run classification on valid policies
classifier = PolicyClassifier()
classified_policies = []
classification_traces = []

for idx, policy in enumerate(valid_policies):
    try:
        classification_result = classifier.forward(policy)
        
        # Parse secondary categories
        secondary_cats = []
        if classification_result.secondary_categories and classification_result.secondary_categories.lower() != "none":
            secondary_cats = [c.strip() for c in classification_result.secondary_categories.split(",")]
        
        # Parse key indicators
        key_inds = []
        if classification_result.key_indicators:
            key_inds = [k.strip() for k in classification_result.key_indicators.split(",")]
        
        # Parse mechanisms
        mechanisms = []
        if classification_result.policy_mechanisms:
            mechanisms = [m.strip() for m in classification_result.policy_mechanisms.split(",")]
        
        # Parse co-benefits
        co_ben = []
        if classification_result.co_benefits and classification_result.co_benefits.lower() != "none":
            co_ben = [b.strip() for b in classification_result.co_benefits.split(",")]
        
        # Add classification fields to the policy
        classified_policy = {
            **policy,  # All original fields including validation
            'primary_category': classification_result.primary_category,
            'secondary_categories': secondary_cats,
            'primary_causal_pathway': classification_result.primary_causal_pathway,
            'key_indicators': key_inds,
            'policy_mechanisms': mechanisms,
            'classification_confidence': classification_result.confidence_score,
            'co_benefits': co_ben
        }
        classified_policies.append(classified_policy)
        
        # Store classification reasoning trace separately
        classification_traces.append({
            'policy_index': idx,
            'policy_statement': policy['policy_statement'],
            'primary_category': classification_result.primary_category,
            'secondary_categories': secondary_cats,
            'classification_reasoning': classification_result.classification_reasoning,
            'confidence_score': classification_result.confidence_score,
            'edge_case_notes': classification_result.edge_case_notes if classification_result.edge_case_notes.lower() != "none" else None,
            'primary_causal_pathway': classification_result.primary_causal_pathway
        })
        
    except Exception as e:
        print(f"Error classifying policy {idx}: {e}")
        # Keep the policy but mark classification as failed
        classified_policy = {
            **policy,
            'primary_category': 'ERROR',
            'classification_confidence': 0.0,
            'classification_error': str(e)
        }
        classified_policies.append(classified_policy)

# Save final classified policies (with all fields from extraction, validation, and classification)
with open("classified_policies_seattle.json", "w") as f:
    json.dump(classified_policies, f, indent=2)

# Save classification reasoning traces separately
with open("classification_traces_seattle.json", "w") as f:
    json.dump(classification_traces, f, indent=2)

# Generate and print classification statistics
category_counts = {}
multi_label_count = 0
total_confidence = 0.0

for policy in classified_policies:
    cat = policy.get('primary_category', 'Unknown')
    category_counts[cat] = category_counts.get(cat, 0) + 1
    
    if policy.get('secondary_categories'):
        multi_label_count += 1
    
    total_confidence += policy.get('classification_confidence', 0.0)

avg_confidence = total_confidence / len(classified_policies) if classified_policies else 0.0

print(f"\n{'='*60}")
print("CLASSIFICATION SUMMARY")
print(f"{'='*60}")
print(f"Total valid policies classified: {len(classified_policies)}")
print(f"Multi-label policies: {multi_label_count}")
print(f"Average classification confidence: {avg_confidence:.3f}")
print(f"\nCategory Distribution:")
for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
    pct = (count / len(classified_policies) * 100) if classified_policies else 0
    print(f"  {cat}: {count} ({pct:.1f}%)")

NameError: name 'valid_policies' is not defined

In [25]:
# Cell: Enhanced Batch Processing Pipeline with CSV Export
# NOTE: Run ALL previous cells first to define DocumentMetadata, PolicyExtractor, PolicyValidator, and PolicyClassifier

import pandas as pd
import json
from pathlib import Path
from typing import List, Dict, Optional

def export_policies_to_csv(classified_policies: List[dict], location_name: str, output_dir: str = "outputs"):
    """
    Export classified policies for a single location to CSV with all fields.
    
    Args:
        classified_policies: List of classified policy dictionaries
        location_name: Name of the location
        output_dir: Directory to save CSV
        
    Returns:
        DataFrame with exported data
    """
    
    rows = []
    for policy in classified_policies:
        row = {
            # Location metadata
            'location': location_name,
            
            # Extraction fields
            'policy_commitment': policy.get('policy_statement', ''),
            'source_quote': policy.get('verbatim_text', ''),
            'has_measurable_target': policy.get('has_quantifiable_target', ''),
            'has_deadline': policy.get('has_timeline', ''),
            'has_legal_mandate': policy.get('has_binding_mechanism', ''),
            'has_geographic_scope': policy.get('has_spatial_specificity', ''),
            'extraction_notes': policy.get('extraction_rationale', ''),
            
            # Validation fields
            'is_actionable': policy.get('final_verdict', ''),
            'soundness_confidence': policy.get('confidence_score', ''),
            'validation_reasoning': policy.get('validation_status', ''),
            'weak_language_detected': '',  # Will be populated if available
            'strong_language_detected': '',  # Will be populated if available
            
            # Classification fields
            'climate_strategy_type': policy.get('primary_category', ''),
            'additional_strategy_types': '; '.join(policy.get('secondary_categories', [])),
            'causal_mechanism': policy.get('primary_causal_pathway', ''),
            'policy_instruments': '; '.join(policy.get('policy_mechanisms', [])),
            'classification_signals': '; '.join(policy.get('key_indicators', [])),
            'classification_confidence': policy.get('classification_confidence', ''),
            'co_benefits_identified': '; '.join(policy.get('co_benefits', [])),
            'classification_notes': ''  # Will be populated if available
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Save to CSV
    csv_file = f"{output_dir}/{location_name}_policies.csv"
    df.to_csv(csv_file, index=False, encoding='utf-8')
    
    print(f"  CSV saved: {csv_file}")
    
    return df


def process_single_document_with_csv(
    location_name: str,
    pdf_path: str,
    country: str,
    state_or_province: Optional[str] = None,
    city: Optional[str] = None,
    output_dir: str = "outputs"
) -> Dict:
    """
    Process a single document and export to both JSON and CSV.
    """
    
    print(f"\n{'='*70}")
    print(f"PROCESSING: {location_name}")
    print(f"File: {pdf_path}")
    print(f"Location: {city or country}, {state_or_province or ''}, {country}")
    print(f"{'='*70}\n")
    
    Path(output_dir).mkdir(exist_ok=True)
    
    try:
        # Step 1: Convert PDF to markdown
        print("Step 1: Converting PDF to markdown...")
        converter = DocumentConverter()
        document = converter.convert(pdf_path)
        markdown_text = document.document.export_to_markdown()
        print(f"✓ Converted {len(markdown_text)} characters\n")
        
        # Step 2: Extract policies
        print("Step 2: Extracting policies...")
        metadata = DocumentMetadata(
            country=country,
            state_or_province=state_or_province,
            city=city
        )
        
        extractor = PolicyExtractor()
        extracted_policies = extractor(
            document_text=markdown_text,
            document_metadata=metadata
        )
        print(f"✓ Extracted {len(extracted_policies)} policy candidates\n")
        
        # Save extracted policies
        extracted_file = f"{output_dir}/{location_name}_extracted_policies.json"
        with open(extracted_file, "w") as f:
            json.dump([policy.dict() for policy in extracted_policies], f, indent=2)
        
        # Step 3: Validate policies
        print("Step 3: Validating policy soundness...")
        validator = PolicyValidator()
        valid_policies = []
        reasoning_traces = []
        
        for idx, policy in enumerate(extracted_policies):
            policy_dict = policy.dict()
            validation_result = validator.forward(policy_dict)
            
            # Store reasoning trace
            reasoning_traces.append({
                'policy_index': idx,
                'policy_statement': policy_dict['policy_statement'],
                'validation_result': validation_result.validation_result,
                'confidence_score': validation_result.confidence_score,
                'reasoning': validation_result.reasoning,
                'weak_signals': validation_result.weak_signals,
                'strong_signals': validation_result.strong_signals,
                'final_verdict': validation_result.final_verdict
            })
            
            # Only keep valid policies
            if validation_result.validation_result == "VALID" and validation_result.final_verdict:
                combined_policy = {
                    **policy_dict,
                    'validation_status': validation_result.validation_result,
                    'confidence_score': validation_result.confidence_score,
                    'final_verdict': validation_result.final_verdict,
                    'weak_signals': validation_result.weak_signals,
                    'strong_signals': validation_result.strong_signals
                }
                valid_policies.append(combined_policy)
        
        print(f"✓ {len(valid_policies)} policies passed validation\n")
        
        # Save validation results
        valid_file = f"{output_dir}/{location_name}_valid_policies.json"
        reasoning_file = f"{output_dir}/{location_name}_reasoning_traces.json"
        with open(valid_file, "w") as f:
            json.dump(valid_policies, f, indent=2)
        with open(reasoning_file, "w") as f:
            json.dump(reasoning_traces, f, indent=2)
        
        # Step 4: Classify valid policies
        print("Step 4: Classifying policies...")
        classifier = PolicyClassifier()
        classified_policies = []
        classification_traces = []
        
        for idx, policy in enumerate(valid_policies):
            try:
                classification_result = classifier.forward(policy)
                
                # Parse lists
                secondary_cats = []
                if classification_result.secondary_categories and classification_result.secondary_categories.lower() != "none":
                    secondary_cats = [c.strip() for c in classification_result.secondary_categories.split(",")]
                
                key_inds = []
                if classification_result.key_indicators:
                    key_inds = [k.strip() for k in classification_result.key_indicators.split(",")]
                
                mechanisms = []
                if classification_result.policy_mechanisms:
                    mechanisms = [m.strip() for m in classification_result.policy_mechanisms.split(",")]
                
                co_ben = []
                if classification_result.co_benefits and classification_result.co_benefits.lower() != "none":
                    co_ben = [b.strip() for b in classification_result.co_benefits.split(",")]
                
                # Combine all fields with comprehensive structure
                classified_policy = {
                    # Core extraction
                    'policy_statement': policy.get('policy_statement', ''),
                    'verbatim_text': policy.get('verbatim_text', ''),
                    'has_quantifiable_target': policy.get('has_quantifiable_target', ''),
                    'has_timeline': policy.get('has_timeline', ''),
                    'has_binding_mechanism': policy.get('has_binding_mechanism', ''),
                    'has_spatial_specificity': policy.get('has_spatial_specificity', ''),
                    'extraction_rationale': policy.get('extraction_rationale', ''),
                    
                    # Validation
                    'validation_status': policy.get('validation_status', ''),
                    'confidence_score': policy.get('confidence_score', ''),
                    'final_verdict': policy.get('final_verdict', ''),
                    'weak_signals': policy.get('weak_signals', ''),
                    'strong_signals': policy.get('strong_signals', ''),
                    
                    # Classification
                    'primary_category': classification_result.primary_category,
                    'secondary_categories': secondary_cats,
                    'primary_causal_pathway': classification_result.primary_causal_pathway,
                    'key_indicators': key_inds,
                    'policy_mechanisms': mechanisms,
                    'classification_confidence': classification_result.confidence_score,
                    'co_benefits': co_ben,
                    'edge_case_notes': classification_result.edge_case_notes if classification_result.edge_case_notes.lower() != "none" else None
                }
                classified_policies.append(classified_policy)
                
                # Store classification trace
                classification_traces.append({
                    'policy_index': idx,
                    'policy_statement': policy['policy_statement'],
                    'primary_category': classification_result.primary_category,
                    'secondary_categories': secondary_cats,
                    'classification_reasoning': classification_result.classification_reasoning,
                    'confidence_score': classification_result.confidence_score,
                    'edge_case_notes': classification_result.edge_case_notes if classification_result.edge_case_notes.lower() != "none" else None,
                    'primary_causal_pathway': classification_result.primary_causal_pathway
                })
                
            except Exception as e:
                print(f"Error classifying policy {idx}: {e}")
                classified_policy = {
                    **policy,
                    'primary_category': 'ERROR',
                    'classification_confidence': 0.0,
                    'classification_error': str(e)
                }
                classified_policies.append(classified_policy)
        
        print(f"✓ Classified {len(classified_policies)} policies\n")
        
        # Save JSON outputs
        classified_file = f"{output_dir}/{location_name}_classified_policies.json"
        classification_trace_file = f"{output_dir}/{location_name}_classification_traces.json"
        with open(classified_file, "w") as f:
            json.dump(classified_policies, f, indent=2)
        with open(classification_trace_file, "w") as f:
            json.dump(classification_traces, f, indent=2)
        
        # Export to CSV
        print("Step 5: Exporting to CSV...")
        csv_df = export_policies_to_csv(classified_policies, location_name, output_dir)
        
        # Calculate statistics
        category_counts = {}
        multi_label_count = 0
        total_classification_confidence = 0.0
        
        for policy in classified_policies:
            cat = policy.get('primary_category', 'Unknown')
            category_counts[cat] = category_counts.get(cat, 0) + 1
            
            if policy.get('secondary_categories'):
                multi_label_count += 1
            
            total_classification_confidence += policy.get('classification_confidence', 0.0)
        
        avg_classification_confidence = total_classification_confidence / len(classified_policies) if classified_policies else 0.0
        
        stats = {
            'location': location_name,
            'total_extracted': len(extracted_policies),
            'passed_validation': len(valid_policies),
            'final_classified': len(classified_policies),
            'validation_rate': len(valid_policies) / len(extracted_policies) if extracted_policies else 0,
            'filtered_out': len(extracted_policies) - len(valid_policies),
            'category_distribution': category_counts,
            'avg_classification_confidence': avg_classification_confidence,
            'multi_label_count': multi_label_count
        }
        
        # Save stats
        stats_file = f"{output_dir}/{location_name}_stats.json"
        with open(stats_file, "w") as f:
            json.dump(stats, f, indent=2)
        
        # Print summary
        print(f"{'='*70}")
        print(f"PROCESSING COMPLETE: {location_name}")
        print(f"{'='*70}")
        print(f"Total extracted: {stats['total_extracted']}")
        print(f"Valid policies: {stats['passed_validation']}")
        print(f"Filtered out: {stats['filtered_out']}")
        print(f"Multi-label: {stats['multi_label_count']}")
        print(f"Avg classification confidence: {avg_classification_confidence:.3f}")
        print(f"\nCategory Distribution:")
        for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
            pct = (count / len(classified_policies) * 100) if classified_policies else 0
            print(f"  {cat}: {count} ({pct:.1f}%)")
        print()
        
        return {
            'status': 'success',
            'stats': stats,
            'classified_policies': classified_policies,
            'csv_df': csv_df,
            'output_files': {
                'extracted': extracted_file,
                'valid': valid_file,
                'reasoning': reasoning_file,
                'classified': classified_file,
                'classification_traces': classification_trace_file,
                'stats': stats_file,
                'csv': f"{output_dir}/{location_name}_policies.csv"
            }
        }
        
    except Exception as e:
        print(f"\n❌ ERROR processing {location_name}: {e}\n")
        return {
            'status': 'error',
            'error': str(e),
            'location': location_name
        }


def process_all_documents_with_csv(documents_config: Dict, output_dir: str = "outputs") -> Dict:
    """
    Process all documents and create individual CSVs + combined CSV.
    """
    
    print(f"\n{'#'*70}")
    print(f"BATCH PROCESSING: {len(documents_config)} DOCUMENTS")
    print(f"{'#'*70}\n")
    
    all_results = {}
    all_dataframes = []
    
    for location_name, config in documents_config.items():
        result = process_single_document_with_csv(
            location_name=location_name,
            pdf_path=config['pdf_path'],
            country=config['country'],
            state_or_province=config.get('state_or_province'),
            city=config.get('city'),
            output_dir=output_dir
        )
        all_results[location_name] = result
        
        if result['status'] == 'success':
            all_dataframes.append(result['csv_df'])
    
    # Combine all CSVs into one
    if all_dataframes:
        print(f"\n{'='*70}")
        print("CREATING COMBINED CSV")
        print(f"{'='*70}\n")
        
        combined_df = pd.concat(all_dataframes, ignore_index=True)
        combined_csv_file = f"{output_dir}/ALL_POLICIES_COMBINED.csv"
        combined_df.to_csv(combined_csv_file, index=False, encoding='utf-8')
        
        print(f"✓ Combined CSV saved: {combined_csv_file}")
        print(f"  Total policies: {len(combined_df)}")
        print(f"\nPolicies by location:")
        print(combined_df['location'].value_counts().to_string())
        print(f"\nPolicies by climate strategy type:")
        print(combined_df['climate_strategy_type'].value_counts().to_string())
    
    # Generate summary report
    print(f"\n{'#'*70}")
    print("BATCH PROCESSING SUMMARY")
    print(f"{'#'*70}\n")
    
    successful = sum(1 for r in all_results.values() if r['status'] == 'success')
    failed = sum(1 for r in all_results.values() if r['status'] == 'error')
    
    print(f"Total documents: {len(documents_config)}")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}\n")
    
    if successful > 0:
        print("Results by location:")
        for location, result in all_results.items():
            if result['status'] == 'success':
                stats = result['stats']
                print(f"\n{location}:")
                print(f"  Extracted: {stats['total_extracted']}")
                print(f"  Valid: {stats['passed_validation']} ({stats['validation_rate']:.1%})")
                print(f"  CSV: {result['output_files']['csv']}")
    
    if failed > 0:
        print("\n\nFailed locations:")
        for location, result in all_results.items():
            if result['status'] == 'error':
                print(f"  {location}: {result['error']}")
    
    # Save overall summary
    summary_file = f"{output_dir}/batch_processing_summary.json"
    summary_data = {
        'total_documents': len(documents_config),
        'successful': successful,
        'failed': failed,
        'combined_csv': combined_csv_file if all_dataframes else None,
        'results': {k: {
            'status': v['status'],
            'stats': v.get('stats'),
            'error': v.get('error'),
            'output_files': v.get('output_files')
        } for k, v in all_results.items()}
    }
    
    with open(summary_file, "w") as f:
        json.dump(summary_data, f, indent=2)
    
    print(f"\nSummary saved to: {summary_file}\n")
    
    return all_results




In [26]:
# Configuration
DOCUMENTS_CONFIG = {
    "seattle": {
        "pdf_path": "pdfs/Seattle.pdf",
        "country": "United States",
        "state_or_province": "Washington",
        "city": "Seattle"
    },
    "las_vegas": {
        "pdf_path": "pdfs/CLV.pdf",
        "country": "United States",
        "state_or_province": "Nevada",
        "city": "Las Vegas"
    },
    "miami_dade": {
        "pdf_path": "pdfs/MIAMI_DADE.pdf",
        "country": "United States",
        "state_or_province": "Florida",
        "city": "Miami-Dade"
    },
    "portugal": {
        "pdf_path": "pdfs/2021 Portugal ADCOM_UNFCCC.pdf",
        "country": "Portugal",
        "state_or_province": None,
        "city": None
    },
    "austin":{
        'pdf_path': "pdfs/Austin_climate_equity.pdf",
        "country": "United States",
        "state_or_province": "Texas",
        "city": "Austin"
    },
    "dakar":{
        'pdf_path': "pdfs/Dakar.pdf",
        "country": "Senegal",
        "state_or_province": None,
        "city": "Dakar"
    },
    "kuwait":{
        'pdf_path': "pdfs/Kuwait-NAP-2019-2030.pdf",
        "country": "Kuwait",
        "state_or_province": None,
        "city": None
    },
    "Chicago":{
        'pdf_path': "pdfs/Chicago-CAP-071822.pdf",
        "country": "United States",
        "state_or_province": "Illinois",
        "city": "Chicago"
    }
}


# Execute batch processing
print("Starting batch processing pipeline...")
results = process_all_documents_with_csv(DOCUMENTS_CONFIG, output_dir="outputs")

print("\n✅ PIPELINE COMPLETE!")
print(f"\nOutputs saved in 'outputs/' directory:")
print(f"  - Individual location JSONs and CSVs")
print(f"  - ALL_POLICIES_COMBINED.csv (all locations in one file)")
print(f"  - batch_processing_summary.json")

2026-01-20 23:32:54,971 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2026-01-20 23:32:54,997 - INFO - Going to convert document batch...
2026-01-20 23:32:54,998 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-01-20 23:32:54,998 - INFO - Auto OCR model selected ocrmac.
2026-01-20 23:32:54,999 - INFO - Accelerator device: 'mps'


Starting batch processing pipeline...

######################################################################
BATCH PROCESSING: 8 DOCUMENTS
######################################################################


PROCESSING: seattle
File: pdfs/Seattle.pdf
Location: Seattle, Washington, United States

Step 1: Converting PDF to markdown...


2026-01-20 23:32:56,334 - INFO - Accelerator device: 'mps'
2026-01-20 23:32:57,404 - INFO - Processing document Seattle.pdf
2026-01-20 23:33:07,655 - INFO - Finished converting document Seattle.pdf in 12.69 sec.
/var/folders/f5/bb79s3gs38q_wp4_2kvwbz9w0000gn/T/ipykernel_21860/1487238006.py:113: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  json.dump([policy.dict() for policy in extracted_policies], f, indent=2)
/var/folders/f5/bb79s3gs38q_wp4_2kvwbz9w0000gn/T/ipykernel_21860/1487238006.py:122: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  policy_dict = policy.dict()
2026/01/20 23:33:07 WARNING dspy.primitives.module: Calling module.forward(...) on 

✓ Converted 49888 characters

Step 2: Extracting policies...
✓ Extracted 18 policy candidates

Step 3: Validating policy soundness...


...

✓ Extracted 31 policy candidates

Step 3: Validating policy soundness...


2026/01/21 00:33:28 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/21 00:33:47 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/21 00:34:01 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/21 00:34:22 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/21 00:34:35 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/21 00:34:55 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyValidator directly is discouraged. Please use module(...) instead.
2026/01/21 00:35:11 WARNING dspy.primitives.module: Calling modu

✓ 23 policies passed validation

Step 4: Classifying policies...


2026/01/21 00:42:33 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyClassifier directly is discouraged. Please use module(...) instead.
2026/01/21 00:42:33 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['policy_statement', 'verbatim_text']. Missing: ['context'].
2026/01/21 00:42:52 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyClassifier directly is discouraged. Please use module(...) instead.
2026/01/21 00:42:52 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['policy_statement', 'verbatim_text']. Missing: ['context'].
2026/01/21 00:43:07 WARNING dspy.primitives.module: Calling module.forward(...) on PolicyClassifier directly is discouraged. Please use module(...) instead.
2026/01/21 00:43:07 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['policy_statement', 'verbatim_text']. Missing: ['context'].
2026/01/21 00:43:29 WARNING dsp

✓ Classified 23 policies

Step 5: Exporting to CSV...
  CSV saved: outputs/Chicago_policies.csv
PROCESSING COMPLETE: Chicago
Total extracted: 31
Valid policies: 23
Filtered out: 8
Multi-label: 11
Avg classification confidence: 0.906

Category Distribution:
  Mitigation: 13 (56.5%)
  Resource Efficiency: 8 (34.8%)
  Nature-Based Solutions: 1 (4.3%)
  Adaptation: 1 (4.3%)


CREATING COMBINED CSV

✓ Combined CSV saved: outputs/ALL_POLICIES_COMBINED.csv
  Total policies: 86

Policies by location:
location
Chicago       23
miami_dade    19
austin        12
seattle       10
dakar          9
portugal       5
las_vegas      4
kuwait         4

Policies by climate strategy type:
climate_strategy_type
Mitigation                40
Resource Efficiency       28
Nature-Based Solutions    10
Adaptation                 8

######################################################################
BATCH PROCESSING SUMMARY
######################################################################

Total document